## 화자 분리 모델로 시간대별 화자 구분하기

In [5]:
import torch
from pyannote.audio import Pipeline
import torchaudio

pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    # use_auth_token='HUGGINGFACEHUB_API_TOKEN'
)

if torch.cuda.is_available():
    pipeline.to(torch.device('cuda'))
    print('cuda is available')
else:
    print('cuda is not available')

cuda is available


In [7]:
waveform, sample_rate = torchaudio.load('./audio/싼기타_비싼기타.mp3')

diarization = pipeline({'waveform': waveform, 'sample_rate': sample_rate})

with open('./audio/싼기타_비싼기타.rttm', 'w', encoding='utf-8') as rttm:
    diarization.speaker_diarization.write_rttm(rttm)

In [8]:
import pandas as pd
rttm_path = './audio/싼기타_비싼기타.rttm'

df_rttm = pd.read_csv(
    rttm_path,
    sep=' ',
    header=None,
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4']
)

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,waveform,1,414.481,2.970,NaN,NaN,SPEAKER_01,NaN,NaN
84,SPEAKER,waveform,1,417.755,3.476,NaN,NaN,SPEAKER_00,NaN,NaN
85,SPEAKER,waveform,1,423.644,0.776,NaN,NaN,SPEAKER_01,NaN,NaN
86,SPEAKER,waveform,1,424.741,3.527,NaN,NaN,SPEAKER_01,NaN,NaN


In [9]:
df_rttm['end'] = df_rttm['start'] + df_rttm['duration']

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204
...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,waveform,1,414.481,2.970,NaN,NaN,SPEAKER_01,NaN,NaN,417.451
84,SPEAKER,waveform,1,417.755,3.476,NaN,NaN,SPEAKER_00,NaN,NaN,421.231
85,SPEAKER,waveform,1,423.644,0.776,NaN,NaN,SPEAKER_01,NaN,NaN,424.420
86,SPEAKER,waveform,1,424.741,3.527,NaN,NaN,SPEAKER_01,NaN,NaN,428.268


In [10]:
df_rttm['number'] = None
df_rttm.at[0, 'number'] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798,0
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388,None
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686,None
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875,None
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204,None
...,...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,waveform,1,414.481,2.970,NaN,NaN,SPEAKER_01,NaN,NaN,417.451,None
84,SPEAKER,waveform,1,417.755,3.476,NaN,NaN,SPEAKER_00,NaN,NaN,421.231,None
85,SPEAKER,waveform,1,423.644,0.776,NaN,NaN,SPEAKER_01,NaN,NaN,424.420,None
86,SPEAKER,waveform,1,424.741,3.527,NaN,NaN,SPEAKER_01,NaN,NaN,428.268,None


In [11]:
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, 'speaker_id'] != df_rttm.at[i-1, 'speaker_id']:
        df_rttm.at[i, 'number'] = df_rttm.at[i-1, 'number'] + 1
    else:
        df_rttm.at[i, 'number'] = df_rttm.at[i-1, 'number']

display(df_rttm.head(10))

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798,0
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388,0
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686,0
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875,0
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204,0
5,SPEAKER,waveform,1,32.414,0.759,NaN,NaN,SPEAKER_01,NaN,NaN,33.173,1
6,SPEAKER,waveform,1,33.545,3.561,NaN,NaN,SPEAKER_01,NaN,NaN,37.106,1
7,SPEAKER,waveform,1,37.628,3.763,NaN,NaN,SPEAKER_01,NaN,NaN,41.391,1
8,SPEAKER,waveform,1,41.611,0.844,NaN,NaN,SPEAKER_00,NaN,NaN,42.455,2
9,SPEAKER,waveform,1,41.645,1.063,NaN,NaN,SPEAKER_01,NaN,NaN,42.708,3


In [12]:
df_rttm_grouped = df_rttm.groupby('number').agg(
    start=pd.NamedAgg(column='start', aggfunc='min'),
    end=pd.NamedAgg(column='end', aggfunc='max'),
    speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
)

display(df_rttm_grouped)

,start,end,speaker_id
number,,,
0,0.993,30.204,SPEAKER_00
1,32.414,41.391,SPEAKER_01
2,41.611,42.455,SPEAKER_00
3,41.645,42.708,SPEAKER_01
4,42.674,44.024,SPEAKER_00
5,45.813,67.109,SPEAKER_01
6,67.227,82.786,SPEAKER_00
7,84.659,102.564,SPEAKER_01
8,103.492,117.532,SPEAKER_00


In [13]:
df_rttm_grouped['duration'] = df_rttm_grouped['end'] - df_rttm_grouped['start']
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

,start,end,speaker_id,duration
0,0.993,30.204,SPEAKER_00,29.211
1,32.414,41.391,SPEAKER_01,8.977
2,41.611,42.455,SPEAKER_00,0.844
3,41.645,42.708,SPEAKER_01,1.063
4,42.674,44.024,SPEAKER_00,1.350
5,45.813,67.109,SPEAKER_01,21.296
6,67.227,82.786,SPEAKER_00,15.559
7,84.659,102.564,SPEAKER_01,17.905
8,103.492,117.532,SPEAKER_00,14.040
9,119.759,138.676,SPEAKER_01,18.917


In [14]:
df_rttm_grouped.to_csv(
    './audio/싼기타_비싼기타_rttm.csv',
    sep=',',
    index=False
)

## 판다스로 문장 분석하고 화자 매칭하기

In [18]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

def whisper_stt(
    audio_file_path: str,
    output_file_path: str = './output.csv'
):
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_id = 'openai/whisper-large-v3-turbo'

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True
    )
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    pipe = pipeline(
        'automatic-speech-recognition',
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        torch_dtype=torch_dtype,
        device=device,
        return_timestamps=True,
        chunk_length_s=10,
        stride_length_s=2,
    )

    result = pipe(audio_file_path)
    df = whisper_to_dataframe(result, output_file_path)

    return result, df

def whisper_to_dataframe(result, output_file_path):
    start_end_text = []

    for chunk in result['chunks']:
        start = chunk['timestamp'][0]
        end = chunk['timestamp'][1]
        text = chunk['text'].strip()
        start_end_text.append([start, end, text])
        df = pd.DataFrame(start_end_text, columns=['start', 'end', 'text'])
        df.to_csv(output_file_path, index=False, sep='|')
    
    return df

In [19]:
if __name__ == '__main__':
    result, df = whisper_stt(
        './audio/싼기타_비싼기타.mp3',
        './audio/싼기타_비싼기타_1.csv'
    )

    print(df)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[tran

      start     end                                               text
0      1.00   11.00  지금부터 저랑 그 역할극을 합시다 역할극을 스탠딩 코미디 스타일로 할 건데 토론을 ...
1     12.00   17.00         그래서 좀 재밌고 자연스럽고 유머러스하게 저랑 대화를 하시면 돼요 자연스럽게
2     17.00   21.46                      그리고 주제는 쌍기타로 전기기타를 시작하는 게 좋으냐
3     21.46   24.36                        아니면 비싼 기타로 전기기타를 시작하는 게 좋으냐
4     24.36   26.00                                      이거를 입장을 나눠가지고
..      ...     ...                                                ...
117  407.00  409.00                                          화난 것 같은데?
118  409.00  414.00                      네, 정말 괜찮습니다. 즐겁게 대화 나누고 있었어요.
119  414.54  417.00                      계속해서 이하고 나누고 싶으시면 편하게 말씀해주세요.
120  417.00  423.00                아니요. 화나셨는데 굳이 더 할 필요 없죠. 그만 잊지 마시죠.
121  423.00  428.96    알겠습니다. 언제든 다시 이야기 나누고 싶으실 때 편하게 말씀해 주세요. 감사합니다.

[122 rows x 3 columns]


In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pyannote.audio import Pipeline

def speaker_diarization(
        audio_file_path: str,
        output_rttm_file_path: str,
        output_csv_file_path: str
    ):
    pipeline = Pipeline.from_pretrained(
        'pyannote/speaker-diarization-3.1',
    )
    
    if torch.cuda.is_available():
        pipeline.to(torch.device('cuda'))
        print('cuda is available')
    else:
        print('cuda is not available')
    waveform, sample_rate = torchaudio.load(audio_file_path)
    diarization_pipeline = pipeline({'waveform': waveform, 'sample_rate': sample_rate})

    with open(output_rttm_file_path, 'w', encoding='utf-8') as rttm:
        diarization_pipeline.speaker_diarization.write_rttm(rttm)
    
    df_rttm = pd.read_csv(
        output_rttm_file_path,
        sep=' ',
        header=None,
        names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4']
    )

    df_rttm['end'] = df_rttm['start'] + df_rttm['duration']

    df_rttm['number'] = None
    df_rttm.at[0, 'number'] = 0

    for i in range(1, len(df_rttm)):
        if df_rttm.at[i, 'speaker_id'] != df_rttm.at[i-1, 'speaker_id']:
            df_rttm.at[i, 'number'] = df_rttm.at[i-1, 'number'] + 1
        else:
            df_rttm.at[i, 'number'] = df_rttm.at[i-1, 'number']
        
    df_rttm_grouped = df_rttm.groupby('')